# 03 序列与结构消融实验

本页保留三组单序列实验，并增加两组结构消融。各组都使用同一批开发患者和五折划分，比较结果以02的完整三序列模型为参考。

| 组别 | 影像输入 | 人工病灶分支 | 四层特征汇总 |
| --- | --- | --- | --- |
| 02完整模型 | T1-FLAIR＋T2＋DWI | 保留 | 注意力 |
| T1、T2、DWI单序列组 | 每组一种序列 | 保留 | 注意力 |
| 去病灶分支 | 三种序列 | 移除 | 注意力 |
| 去注意力 | 三种序列 | 保留 | 平均 |

两组新模型分别重新训练，不使用02已经训练好的权重。三组单序列已有结果可以直接读取，只开启需要运行的新组即可。

四层缓存、图像增强、ImageNet初始化、AdamW、学习率、损失和早停沿用原设置。去病灶后特征从2304维变为2048维，注意力输入和分类头随之匹配；去注意力后改为四层特征取平均。两组均不另外加入临床融合。

上传时同名替换这一本即可，保留原配置、其他六本Notebook和outputs目录。前面的更新登记单元会处理这次03版本变化。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    settings = path.parent / "experiment_settings.json"
    if settings.is_file():
        overlay = json.loads(settings.read_text(encoding="utf-8"))
        allowed = {"annotation_scope", "optimized", "radiomics", "segmentation", "report_version"}
        if set(overlay) - allowed:
            raise ValueError("experiment_settings.json 只放实验参数，机器路径仍在 config.json。")
        for key, value in overlay.items():
            if isinstance(value, dict):
                cfg[key] = {**cfg.get(key, {}), **value}
            else:
                cfg[key] = value
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg




def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics" / experiment_name(cfg, "radiomics", "selected2d_v1"),
        "explain": root / "explain", "segmentation": root / "segmentation" / experiment_name(cfg, "segmentation", "selected2d_v1"),
        "optimized": root / "optimized" / experiment_name(cfg, "optimized", "resnet_v1"),
        "reports": root / "reports" / safe_name(cfg.get("report_version", "comparison_v2")),
    }




def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")




def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))




def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()




def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result




def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "resnet_baseline_and_optimized_selected_slices", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)




def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table




def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)




def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)




def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    manifest["artifacts"] = {portable_name(k): v for k, v in manifest["artifacts"].items()}
    for key in ("cache", "cohort"):
        manifest[key] = portable_name(manifest[key])
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")


    # Notebook hashes are provenance, not data identity. Model contracts are checked separately.


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本操作需要原分辨率影像，请在数据电脑执行；导入结果和生成报告无需原影像。")




def safe_name(value):
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError("实验名称只使用字母、数字、下划线或短横线。")
    return value




def experiment_name(cfg, stage, default):
    return safe_name(cfg.get(stage, {}).get("experiment", default))




def portable_name(value):
    from pathlib import PurePosixPath
    name = str(value).replace("\\", "/")
    path = PurePosixPath(name)
    if path.is_absolute() or not path.parts or any(p in ("..", ".") or ":" in p for p in path.parts):
        raise ValueError(f"数据包必须使用包内相对路径：{value}")
    return path.as_posix()




def contained_file(root, name):
    root = Path(root).resolve()
    target = (root / portable_name(name)).resolve()
    target.relative_to(root)
    return target




def cohort_identity(cfg):
    """Patient identity independent of local directories, CSV formatting and row order."""
    import pandas as pd
    table = read_cohort(cfg)
    columns = ["mr", "label", "split", "development_fold", "max_layer", "moca"] + CLINICAL
    if not set(columns) <= set(table):
        raise ValueError("病例清单缺少患者、结局、分组、取层或临床列。")
    rows = []
    for _, row in table.sort_values("mr").iterrows():
        values = {}
        for key in columns:
            value = row[key]
            values[key] = (None if pd.isna(value) else str(value) if key in ("mr", "split") else format(float(value), ".12g"))
        rows.append(values)
    payload = {"schema": 1, "patients": rows}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()




def definition_contract(cfg, prefix, names):
    import ast
    files = list(cfg["_root"].glob(prefix + "_*.ipynb"))
    if len(files) != 1:
        raise ValueError(f"找不到唯一的 {prefix} Notebook。")
    notebook = json.loads(files[0].read_text(encoding="utf-8"))
    definitions = {}
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names:
                if node.name in definitions:
                    raise ValueError(f"重复定义：{node.name}")
                definitions[node.name] = hashlib.sha256(ast.dump(node, include_attributes=False).encode()).hexdigest()
    if set(definitions) != set(names):
        raise ValueError("模型或处理流程定义不齐全。")
    return definitions




def verify_transfer(cfg, directory, kind=None):
    directory = Path(directory).resolve()
    manifest = json.loads((directory / "transfer_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("schema") != 1 or manifest.get("kind") not in ("radiomics_features", "radiomics_complete", "segmentation_results", "segmentation_weights"):
        raise ValueError("不支持的数据包类型或版本。")
    if kind and manifest["kind"] != kind:
        raise ValueError("数据包类型不匹配。")
    if manifest["cohort_identity"] != cohort_identity(cfg):
        raise ValueError("两台机器的患者、标签、临床数据、取层或开发折不一致。")
    if manifest["native_source_identity"] != native_source_identity(cfg):
        raise ValueError("数据包与分类缓存记录的原始影像或人工标注不是同一版本。")
    if manifest["annotation_scope"] != "selected_slices" or cfg["annotation_scope"] != "selected_slices":
        raise ValueError("当前流程只接受四层标注协议。")
    for name, checksum in manifest["artifacts"].items():
        file = contained_file(directory, name)
        if not file.is_file() or digest(file) != checksum:
            raise ValueError(f"数据包文件缺失或传输后发生变化：{name}")
    return manifest




def export_transfer(cfg, source, filenames, destination, kind, protocol):
    import shutil
    source, destination = Path(source).resolve(), Path(destination).resolve()
    fresh(destination)
    mapping = {}
    for name in filenames:
        name = portable_name(name)
        file = contained_file(source, name)
        if not file.is_file():
            raise FileNotFoundError(file)
        mapping[name] = digest(file)
    destination.mkdir(parents=True)
    for name in mapping:
        target = contained_file(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), target)
    dump_json(destination / "transfer_manifest.json", {
        "schema": 1, "kind": kind, "cohort_identity": cohort_identity(cfg), "native_source_identity": native_source_identity(cfg),
        "annotation_scope": cfg["annotation_scope"], "protocol": protocol,
        "artifacts": mapping, "source_run_id": cfg["run_id"],
        "notebook_definitions_sha256": source_snapshot(cfg)})
    verify_transfer(cfg, destination, kind)
    return destination




def import_transfer(cfg, source, stage):
    import shutil
    source = Path(source).resolve()
    manifest = verify_transfer(cfg, source)
    allowed = {"radiomics": ("radiomics_features", "radiomics_complete"),
               "segmentation": ("segmentation_results",), "segmentation_weights": ("segmentation_weights",)}
    if stage not in allowed or manifest["kind"] not in allowed[stage]:
        raise ValueError("数据包与接收阶段不匹配。")
    expected_protocol = "selected_four_slices_2d_pixels_v1"
    if manifest["protocol"] != expected_protocol:
        raise ValueError("数据包的二维取层/像素空间处理协议不匹配。")
    target = paths(cfg)[stage] if stage != "segmentation_weights" else paths(cfg)["segmentation"].with_name(paths(cfg)["segmentation"].name + "_imported_model")
    fresh(target)
    target.mkdir(parents=True)
    for name in manifest["artifacts"]:
        dest = contained_file(target, name)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), dest)
    shutil.copy2(source / "transfer_manifest.json", target / "transfer_manifest.json")
    verify_transfer(cfg, target)
    return target




def native_source_identity(cfg):
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    patients = read_cohort(cfg).mr.tolist()
    files = IMAGE_FILES + ["Untitled.nii.gz"]
    selected = {mr: {name: audit["patients"][mr]["files"][name]["sha256"] for name in files} for mr in patients}
    return hashlib.sha256(json.dumps(selected, sort_keys=True).encode()).hexdigest()




## 读取现有数据与消融结果

数据包继续核对患者和文件内容。原来的单文件版本登记已由各阶段的实现校验取代，更新其他Notebook不会使本页结果失效。去病灶和去注意力两组仍使用原训练设置，不能与优化版的组件消融混为一谈。


In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
STRUCTURE_EXPERIMENT = "structure_v1"
print("实验：", cfg["run_id"])
print("原结果目录：", p["run"])
print("结构消融结果：", p["run"] / "structure_ablation" / STRUCTURE_EXPERIMENT)


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。

ROC、混淆矩阵和重采样工具用于整理各组指标，并比较同一批患者上的预测差异。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


In [ ]:
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils import resample


## 训练工具与固定设置

随机种子、每批患者数和计算设备沿用原代码。单序列实验通过 `SINGLE_SEQUENCES` 选取缓存通道；新增两组结构实验始终使用全部三种影像序列。

新组使用原来的batch=16，影像骨干学习率为1e-4，其余保留模块为5e-4，AdamW的weight decay为1e-4。每折最多30轮，连续8轮留出折AUC没有改善则停止，梯度裁剪为1.0。


In [ ]:
from __future__ import annotations
import copy
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

SEED = 20260922
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SINGLE_SEQUENCES = {"t1": 0, "t2": 1, "dwi": 2}
CACHE = OUT_DIR = TEMPORAL_OUT = TEMPORAL_SPLIT = None
PREP_BASE = PREP_OUT = ADAPTER_XLSX = None


## 连接输入与输出目录

`configure_baseline` 把配置中确定的文件位置传给后续函数，包括影像缓存、临床表、时间分组表和模型输出目录。后面调用数据处理或模型函数时，就会使用同一组路径。


In [ ]:
from __future__ import annotations

def configure_baseline(cache, full_cv, temporal, adapter, patients_root=None):
    """Configure only input/output locations; do not alter the training recipe."""
    global CACHE, OUT_DIR, TEMPORAL_OUT, TEMPORAL_SPLIT
    global PREP_BASE, PREP_OUT, ADAPTER_XLSX
    CACHE, OUT_DIR, TEMPORAL_OUT = Path(cache), Path(full_cv), Path(temporal)
    TEMPORAL_SPLIT, ADAPTER_XLSX = TEMPORAL_OUT / "temporal_split.csv", Path(adapter)
    PREP_BASE, PREP_OUT = Path(patients_root) if patients_root else None, CACHE


## 病灶分支、数据增强与概率输出

`MaskEncoder` 将每层的三类病灶掩模编码为 256 维特征，三组单序列实验使用相同的病灶分支。

`augment_batch` 对同一患者的四层影像和掩模同步进行几何变换，再对影像加入亮度和噪声变化。`seed_everything` 设置随机数来源，`predict_proba` 则切换到评估状态，分批返回患者的阳性概率。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_manual_cv.py :: seed_everything
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# 直接复制：code/train_resnet50_manual_cv.py :: MaskEncoder
class MaskEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x.float())


# 直接复制：code/train_resnet50_manual_cv.py :: augment_batch
def augment_batch(images: torch.Tensor, masks: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    out_images = images.clone()
    out_masks = masks.clone()
    B, S = images.shape[:2]
    for b in range(B):
        angle = random.uniform(-6.0, 6.0)
        translate = (random.uniform(-6.0, 6.0), random.uniform(-6.0, 6.0))
        scale = random.uniform(0.96, 1.04)
        for s in range(S):
            out_images[b, s] = affine(
                images[b, s],
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            out_masks[b, s] = affine(
                masks[b, s].float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).byte()

    jitter = 0.9 + 0.2 * torch.rand(B, 1, 1, 1, 1, device=out_images.device)
    out_images = out_images * jitter
    if random.random() < 0.5:
        out_images = out_images + torch.randn_like(out_images) * 0.008
    out_images = out_images.clamp(0.0, 1.0)
    return out_images, out_masks


# 直接复制：code/train_resnet50_manual_cv.py :: predict_proba
@torch.no_grad()
def predict_proba(
    model: nn.Module,
    X: torch.Tensor,
    M: torch.Tensor,
    batch_size: int = BATCH_SIZE,
) -> np.ndarray:
    model.eval()
    outs = []
    for i in range(0, len(X), batch_size):
        xb = X[i : i + batch_size].to(DEVICE)
        mb = M[i : i + batch_size].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            outs.append(torch.sigmoid(model(xb, mb)).float().cpu().numpy())
    return np.concatenate(outs)


## 单通道 ResNet 与每折训练

`SingleSequenceDualBranchModel` 把 ResNet50 的第一层卷积改为单通道输入，初始化时对预训练卷积的三个输入通道权重取均值。其余部分继续按每层 2048 维影像特征、256 维病灶特征和四层注意力汇总的方式计算。

`train_single_sequence_fold` 完成一折训练，使用 AdamW、加权二元交叉熵和数据增强。最多训练 30 轮，连续 8 轮留出折 AUC 未改善时停止，返回该折 AUC 最佳时的模型。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_single_sequence_manual_cv.py :: SingleSequenceDualBranchModel
class SingleSequenceDualBranchModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        old_conv = backbone.conv1
        conv1 = nn.Conv2d(
            1,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            conv1.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        backbone.conv1 = conv1
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = MaskEncoder()
        self.attention = nn.Sequential(
            nn.Linear(2304, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2304),
            nn.Dropout(0.4),
            nn.Linear(2304, 1),
        )

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        im = images.view(B * S, C, H, W)
        image_feat = self.image_features(im)
        image_feat = torch.flatten(image_feat, 1)
        image_feat = image_feat.view(B, S, -1)

        mask = masks.view(B * S, 3, H, W)
        mask_feat = self.mask_features(mask)
        mask_feat = mask_feat.view(B, S, -1)

        feat = torch.cat([image_feat, mask_feat], dim=-1)
        attn = self.attention(feat).squeeze(-1)
        attn = torch.softmax(attn, dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)


# 直接复制：code/train_resnet50_single_sequence_manual_cv.py :: train_fold
def train_single_sequence_fold(
    Xtr: torch.Tensor,
    Mtr: torch.Tensor,
    ytr: torch.Tensor,
    Xva: torch.Tensor,
    Mva: torch.Tensor,
    yva: torch.Tensor,
    fold: int,
) -> SingleSequenceDualBranchModel:
    model = SingleSequenceDualBranchModel().to(DEVICE)
    optimizer = torch.optim.AdamW(
        [
            {"params": model.image_features.parameters(), "lr": 1e-4},
            {
                "params": list(model.mask_features.parameters())
                + list(model.attention.parameters())
                + list(model.head.parameters()),
                "lr": 5e-4,
            },
        ],
        weight_decay=1e-4,
    )
    n_pos = int((ytr > 0.5).sum())
    n_neg = len(ytr) - n_pos
    pos_weight = torch.tensor([max(n_neg / max(n_pos, 1), 0.5)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_auc, best_state, patience = 0.0, None, 0

    for epoch in range(1, 31):
        t0 = time.time()
        model.train()
        perm = torch.randperm(len(Xtr))
        epoch_loss = 0.0
        for i in range(0, len(Xtr), BATCH_SIZE):
            bi = perm[i : i + BATCH_SIZE]
            xb, mb = Xtr[bi], Mtr[bi]
            xb, mb = augment_batch(xb, mb)
            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)
            yb = ytr[bi].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                loss = criterion(model(xb, mb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += float(loss.detach()) * len(bi)
        epoch_loss /= len(Xtr)

        val_pred = predict_proba(model, Xva, Mva)
        val_auc = float(roc_auc_score(yva.numpy(), val_pred))
        if val_auc > best_auc:
            best_auc, best_state, patience = val_auc, copy.deepcopy(model.state_dict()), 0
        else:
            patience += 1
        if epoch % 3 == 0 or patience >= 8:
            print(
                f"fold {fold} epoch {epoch:02d} loss={epoch_loss:.4f} "
                f"val_auc={val_auc:.4f} best={best_auc:.4f} time={time.time()-t0:.1f}s",
                flush=True,
            )
        if patience >= 8:
            break

    model.load_state_dict(best_state)
    return model


## 去病灶分支与去注意力的模型

`ImageOnlyAttentionModel` 只使用ResNet提取的2048维影像特征，保留切片注意力和分类头，不读取掩模内容进行预测。它的接口仍接收掩模参数，便于共用原训练与增强函数。

`MeanPoolingDualBranchModel` 保留2048维影像特征与256维病灶特征，将四层2304维特征直接取平均，不再使用可学习的注意力层。分类头与完整模型相同。


In [ ]:
from __future__ import annotations


class ImageOnlyAttentionModel(nn.Module):
    """Three image sequences and slice attention, without a mask encoder."""
    def __init__(self) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        # Parameter-free placeholder preserves the original optimizer grouping.
        self.mask_features = nn.Identity()
        self.attention = nn.Sequential(
            nn.Linear(2048, 256), nn.Tanh(), nn.Dropout(0.2), nn.Linear(256, 1),
        )
        self.head = nn.Sequential(nn.LayerNorm(2048), nn.Dropout(0.4), nn.Linear(2048, 1))

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        image_feat = self.image_features(images.view(B * S, C, H, W))
        feat = torch.flatten(image_feat, 1).view(B, S, -1)
        attn = torch.softmax(self.attention(feat).squeeze(-1), dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)








class MeanPoolingDualBranchModel(nn.Module):
    """Three image sequences and masks, with uniform averaging over four slices."""
    def __init__(self) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = MaskEncoder()
        self.attention = nn.Identity()
        self.head = nn.Sequential(nn.LayerNorm(2304), nn.Dropout(0.4), nn.Linear(2304, 1))

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        image_feat = self.image_features(images.view(B * S, C, H, W))
        image_feat = torch.flatten(image_feat, 1).view(B, S, -1)
        mask_feat = self.mask_features(masks.view(B * S, 3, H, W)).view(B, S, -1)
        feat = torch.cat([image_feat, mask_feat], dim=-1)
        pooled = feat.mean(dim=1)
        return self.head(pooled).squeeze(-1)








## 沿用原优化器、损失和每折训练

优化器与加权二元交叉熵直接沿用02的函数。移除的模块没有训练参数，其余模块仍归入原来的学习率分组。

`train_structure_fold` 使用原开发训练循环，只根据组别构造相应模型。数据增强、混合精度、梯度裁剪、早停和最佳epoch选择保持原样，每折返回最佳模型与轮数。


In [ ]:
from __future__ import annotations


def make_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    return torch.optim.AdamW(
        [
            {"params": model.image_features.parameters(), "lr": 1e-4},
            {
                "params": list(model.mask_features.parameters())
                + list(model.attention.parameters())
                + list(model.head.parameters()),
                "lr": 5e-4,
            },
        ],
        weight_decay=1e-4,
    )



def make_criterion(y: np.ndarray | torch.Tensor) -> nn.BCEWithLogitsLoss:
    y_np = np.asarray(y)
    n_pos = int((y_np > 0.5).sum())
    n_neg = len(y_np) - n_pos
    pos_weight = torch.tensor([max(n_neg / max(n_pos, 1), 0.5)], device=DEVICE)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)



In [ ]:
from __future__ import annotations


def train_structure_fold(
    variant: str,
    Xtr: torch.Tensor,
    Mtr: torch.Tensor,
    ytr: torch.Tensor,
    Xva: torch.Tensor,
    Mva: torch.Tensor,
    yva: torch.Tensor,
    fold: int,
) -> tuple[nn.Module, int]:
    model_type = {"no_mask": ImageOnlyAttentionModel, "no_attention": MeanPoolingDualBranchModel}[variant]
    model = model_type().to(DEVICE)
    optimizer = make_optimizer(model)
    criterion = make_criterion(ytr.numpy())
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    best_auc, best_state, best_epoch, patience = 0.0, None, 1, 0

    for epoch in range(1, 31):
        model.train()
        perm = torch.randperm(len(Xtr))
        epoch_loss = 0.0
        for i in range(0, len(Xtr), BATCH_SIZE):
            bi = perm[i : i + BATCH_SIZE]
            xb, mb = Xtr[bi], Mtr[bi]
            xb, mb = augment_batch(xb, mb)
            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)
            yb = ytr[bi].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                loss = criterion(model(xb, mb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += float(loss.detach()) * len(bi)

        val_pred = predict_proba(model, Xva, Mva)
        val_auc = float(roc_auc_score(yva.numpy(), val_pred))
        if val_auc > best_auc:
            best_auc, best_state, best_epoch, patience = val_auc, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            patience += 1
        if epoch % 3 == 0 or patience >= 8:
            print(
                f"fold {fold} epoch {epoch:02d} loss={epoch_loss/len(Xtr):.4f} "
                f"val_auc={val_auc:.4f} best={best_auc:.4f}",
                flush=True,
            )
        if patience >= 8:
            break

    model.load_state_dict(best_state)
    return model, best_epoch








## 指标计算与同一患者上的配对比较

前两个函数负责从开发预测选择分类阈值，并计算 AUC、准确率、敏感度和特异度。`paired_comparison` 先按 MR 编号对齐两个模型的患者和标签，再在每次抽样中同时取出两个模型对同一批患者的概率，计算 AUC 差值。

输出的差值按“模型 A 减模型 B”定义，同时给出重采样区间和经验 p 值。配对方式保留了两个模型在同一患者上预测的关联。


In [ ]:
from __future__ import annotations

def threshold_from_development(y, probability):
    fpr, tpr, thresholds = roc_curve(y, probability)
    return float(thresholds[np.argmax(tpr - fpr)])







def classification_metrics(y, probability, threshold, n_bootstrap=1000):
    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)
    if len(y) != len(probability) or set(y) != {0, 1} or not np.isfinite(probability).all():
        raise ValueError("标签需包含两类，概率必须有限且与患者一一对应。")
    if ((probability < 0) | (probability > 1)).any():
        raise ValueError("概率不在[0,1]。")
    tn, fp, fn, tp = confusion_matrix(y, probability >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "auc": float(roc_auc_score(y, probability)), "threshold": float(threshold),
              "accuracy": float((tp + tn) / len(y)), "sensitivity": float(tp / (tp + fn)),
              "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(y, probability)),
              "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
    if n_bootstrap:
        values = []
        for seed in range(n_bootstrap):
            idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
            values.append(roc_auc_score(y[idx], probability[idx]))
        result.update(auc_ci_low=float(np.percentile(values, 2.5)), auc_ci_high=float(np.percentile(values, 97.5)))
    return result







def paired_comparison(a: pd.DataFrame, b: pd.DataFrame, n_bootstrap=1000):
    for frame in (a, b):
        if not frame["mr"].is_unique or set(frame["label"]) != {0, 1}:
            raise ValueError("比较需要唯一患者编号与两类结局。")
    if set(a["mr"]) != set(b["mr"]):
        raise ValueError("两组患者不完全一致；不静默取交集。")
    merged = a.merge(b, on="mr", suffixes=("_a", "_b"), validate="one_to_one")
    if not (merged.label_a == merged.label_b).all():
        raise ValueError("同一患者的标签不一致。")
    y, pa, pb = merged.label_a.to_numpy(), merged.probability_a.to_numpy(), merged.probability_b.to_numpy()
    if not np.isfinite(pa).all() or not np.isfinite(pb).all():
        raise ValueError("预测包含非有限值。")
    differences = []
    for seed in range(n_bootstrap):
        idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
        differences.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.asarray(differences)
    return {"auc_difference": float(roc_auc_score(y, pa) - roc_auc_score(y, pb)),
            "ci_low": float(np.percentile(d, 2.5)), "ci_high": float(np.percentile(d, 97.5)),
            "empirical_p": min(1.0, 2 * min(float((d <= 0).mean()), float((d >= 0).mean())))}







## 依次完成三组单序列实验

`run_single_sequence_ablation` 从缓存中取出开发患者，按 T1-FLAIR、T2、DWI 依次训练。每组截取一个影像通道，病灶掩模保持三个通道，并按已保存的五折划分调用训练函数。

每个序列目录下保存五个折模型和开发折外概率，汇总表保存各折平均 AUC、合并 AUC 及阈值相关指标，供后面与三序列模型比较。


In [ ]:
from __future__ import annotations

def configure(cfg):
    p = paths(cfg)
    configure_baseline(p["cache"], p["full_cv"], p["temporal"], p["adapter"], cfg.get("patients_root"))
    return p







def run_single_sequence_ablation(cfg):
    """Same original single-sequence model and train_fold, restricted to development."""
    import torch
    from sklearn.metrics import roc_auc_score
    p = configure(cfg)
    fresh(p["single"])
    dev = development(cfg)
    indices = dev.cache_index.to_numpy(dtype=int)
    with np.load(p["cache"], allow_pickle=False) as cache:
        X = torch.from_numpy(cache["images"][indices].astype(np.float32))
        M = torch.from_numpy(cache["masks"][indices].astype(np.uint8))
    y = dev.label.to_numpy(dtype=int)
    summaries = []
    for sequence, channel in SINGLE_SEQUENCES.items():
        seed_everything(SEED)
        output = p["single"] / sequence
        output.mkdir(parents=True)
        x = X[:, :, channel:channel + 1].contiguous()
        oof, aucs = np.zeros(len(y), dtype=np.float32), []
        for fold in range(1, 6):
            va = np.flatnonzero(dev.development_fold.to_numpy() == fold)
            tr = np.flatnonzero(dev.development_fold.to_numpy() != fold)
            model = train_single_sequence_fold(x[tr], M[tr], torch.tensor(y[tr], dtype=torch.float32),
                                      x[va], M[va], torch.tensor(y[va], dtype=torch.float32), fold)
            oof[va] = predict_proba(model, x[va], M[va])
            aucs.append(float(roc_auc_score(y[va], oof[va])))
            torch.save(model.state_dict(), output / f"fold_{fold}.pt")
        threshold = threshold_from_development(y, oof)
        table = dev[["mr", "label", "development_fold"]].copy()
        table["probability"] = oof
        table.to_csv(output / "development_oof.csv", index=False)
        summaries.append({"sequence": sequence, "mean_fold_auc": float(np.mean(aucs)),
                          **classification_metrics(y, oof, threshold, n_bootstrap=0)})
    summary = pd.DataFrame(summaries)
    summary.to_csv(p["single"] / "summary.csv", index=False)
    record(cfg, "single_sequence", {"scope": "development", "manual_mask_channels": 3})
    return summary







In [ ]:
from __future__ import annotations

def structure_code_contract(cfg):
    return definition_contract(cfg, "03", list(LEGACY_STRUCTURE_CONTRACT))






LEGACY_STRUCTURE_CONTRACT = {'seed_everything': '2f16b64e559c5d1efa0def1b98e9123cfdebaaccc483dfd396ab65d88b745761', 'MaskEncoder': 'b0a35f2673a19f24a5f492a757a65894b572a9819996d2aabb9572cc89b2de04', 'augment_batch': 'fff7579e369d643e92dbf59bed563b7b2b94344c48ce36598044feb088a19ae2', 'predict_proba': '94646b415f42d60d4a16416a05f26afed9656dd0fba15b09323c50f226898663', 'ImageOnlyAttentionModel': '782e3393d40f2625b0ef20bdf3f0634b54fdba5c32c2dcdf197ef23dba4ddfe7', 'MeanPoolingDualBranchModel': 'b32bc8847e1e040c085b0241be01197d45a2d849ba8bd45e15a5a577683cd1b0', 'make_optimizer': 'f36a1e6e87e23c160bca103ba98ea79aac68811c9a60324b91892a8413ae3730', 'make_criterion': '86e189018025dbbc4d0376bf5ec452ccb415b76507a5c4733d985901fd513528', 'train_structure_fold': '9208f3daa7e0cdac9cc1424cc5613164f3a07966c7e2dec362f71fb93379d160'}


## 结构消融的患者读取、训练与结果汇总

下面先核对缓存里的患者与结局，再确认折号与02的原始五折规则一致。每组从相同随机种子开始训练五折，保存每位患者的折外概率、各折AUC、最佳轮数和模型参数量。

输入文件和代码版本随结果保存。汇总与配对比较会读取已有结果，因此可以先运行一组，再补另一组；不需要重新训练已经完成的单序列模型。


In [ ]:
from __future__ import annotations


def structure_output_directory(cfg, experiment="structure_v1"):
    if not isinstance(experiment, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", experiment):
        raise ValueError("结构消融实验名只使用字母、数字、下划线或短横线。")
    return paths(cfg)["run"] / "structure_ablation" / experiment








def structure_input_fingerprints(cfg):
    p = paths(cfg)
    return {"cache_sha256": digest(p["cache"]), "cohort_sha256": digest(p["cohort"]),
            "temporal_split_sha256": digest(p["temporal"] / "temporal_split.csv")}








def validate_development_predictions(table, dev):
    if not {"mr", "label", "probability"} <= set(table.columns) or not table.mr.is_unique:
        raise ValueError("开发预测缺少必要字段或患者编号重复。")
    if set(table.mr) != set(dev.mr):
        raise ValueError("开发预测与当前开发患者不完全一致。")
    aligned = table.set_index("mr").loc[dev.mr]
    if not np.array_equal(aligned.label.to_numpy(), dev.label.to_numpy()):
        raise ValueError("开发预测的结局与患者清单不一致。")
    probability = aligned.probability.to_numpy(dtype=float)
    if not np.isfinite(probability).all() or np.any((probability < 0) | (probability > 1)):
        raise ValueError("开发预测概率必须是[0,1]内的有限值。")
    if "development_fold" in aligned and not np.array_equal(
            aligned.development_fold.to_numpy(), dev.development_fold.to_numpy()):
        raise ValueError("开发预测的折号与患者清单不同。")








def run_structure_ablation(cfg, variant, experiment="structure_v1"):
    if variant not in {"no_mask", "no_attention"}:
        raise ValueError("结构消融只支持 no_mask 或 no_attention。")
    p, dev = paths(cfg), development(cfg)
    output = structure_output_directory(cfg, experiment) / variant
    if output.exists():
        raise FileExistsError(f"该组结果已存在：{output}。可关闭对应开关读取结果；如需另跑，修改 STRUCTURE_EXPERIMENT。")
    fingerprints = structure_input_fingerprints(cfg)
    with np.load(p["cache"], allow_pickle=False) as cache:
        indices = dev.cache_index.to_numpy(dtype=int)
        if not np.array_equal(cache["mrs"].astype(str)[indices], dev.mr.to_numpy()):
            raise ValueError("开发患者与缓存索引不一致。")
        if not np.array_equal(cache["labels"][indices], dev.label.to_numpy()):
            raise ValueError("缓存结局与开发患者清单不一致。")
        X = torch.from_numpy(cache["images"][indices].astype(np.float32))
        M = torch.from_numpy(cache["masks"][indices].astype(np.uint8))
    if tuple(X.shape[1:]) != (4, 3, 224, 224) or M.shape != X.shape:
        raise ValueError("结构消融沿用四层、三序列、三类掩模、224×224缓存。")
    y = dev.label.to_numpy(dtype=int)
    if set(y) != {0, 1} or np.bincount(y, minlength=2).min() < 5:
        raise ValueError("开发患者需要两类结局，且每类至少5例。")
    folds = dev.development_fold.to_numpy(dtype=int)
    expected_folds = np.zeros(len(dev), dtype=int)
    for fold, (_, valid) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y), 1):
        expected_folds[valid] = fold
    if not np.array_equal(folds, expected_folds):
        raise ValueError("当前折号与02的原始五折规则不一致，请核对患者顺序和清单。")
    baseline_oof = p["temporal"] / "development_oof.csv"
    if baseline_oof.exists():
        baseline = pd.read_csv(baseline_oof, dtype={"mr": str}).rename(columns={"dev_oof_prob": "probability"})
        validate_development_predictions(baseline, dev)
    output.mkdir(parents=True)
    snapshot = source_snapshot(cfg)
    notebook_name = "03_任务3_原单序列消融整合.ipynb"
    training = {"variant": variant, "experiment": experiment, "input_files": fingerprints,
        "notebook03_definitions_sha256": snapshot[notebook_name], "code_contract": structure_code_contract(cfg), "seed": SEED, "batch_size": BATCH_SIZE,
        "optimizer": "AdamW", "backbone_lr": 1e-4, "remaining_modules_lr": 5e-4, "weight_decay": 1e-4,
        "max_epochs": 30, "patience": 8, "gradient_clip_norm": 1.0,
        "loss": "original BCEWithLogitsLoss with training-fold pos_weight", "augmentation": "original augment_batch",
        "image_channels": 3, "mask_channels_used": 0 if variant == "no_mask" else 3,
        "slice_pooling": "attention" if variant == "no_mask" else "mean",
        "scope": "development_only", "selection": "same validation fold selects epoch and reports AUC"}
    dump_json(output / "training_config.json", training)
    seed_everything(SEED)
    oof = np.zeros(len(y), dtype=np.float32)
    fold_rows = []
    for fold in range(1, 6):
        valid, train = np.flatnonzero(folds == fold), np.flatnonzero(folds != fold)
        model, best_epoch = train_structure_fold(
            variant, X[train], M[train], torch.from_numpy(y[train].astype(np.float32)),
            X[valid], M[valid], torch.from_numpy(y[valid].astype(np.float32)), fold,
        )
        oof[valid] = predict_proba(model, X[valid], M[valid])
        auc = float(roc_auc_score(y[valid], oof[valid]))
        fold_rows.append({"fold": fold, "n_train": len(train), "n_valid": len(valid), "auc": auc,
                          "best_epoch": int(best_epoch), "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad)})
        torch.save(model.state_dict(), output / f"fold_{fold}.pt")
        print(f"{variant}: fold {fold}, AUC={auc:.4f}, best_epoch={best_epoch}", flush=True)
        del model
    if fingerprints != structure_input_fingerprints(cfg) or training["code_contract"] != structure_code_contract(cfg):
        raise ValueError("训练期间输入或Notebook定义发生变化；保留折模型供排查，不发布汇总结果。")
    threshold = threshold_from_development(y, oof)
    if not np.isfinite(threshold):
        raise ValueError("该组未得到有限的Youden阈值，请检查开发预测。")
    table = dev[["mr", "label", "development_fold"]].copy()
    table["probability"] = oof
    validate_development_predictions(table, dev)
    table.to_csv(output / "development_oof.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(output / "fold_metrics.csv", index=False)
    aucs = [row["auc"] for row in fold_rows]
    metrics = {"variant": variant, "experiment": experiment, "mean_fold_auc": float(np.mean(aucs)),
        "std_fold_auc": float(np.std(aucs)), "fold_aucs": aucs, "best_epochs": [row["best_epoch"] for row in fold_rows],
        **classification_metrics(y, oof, threshold, n_bootstrap=0)}
    metrics["pooled_auc"] = metrics["auc"]
    dump_json(output / "metrics.json", metrics)
    record(cfg, "structure_" + experiment + "_" + variant, {
        "protocol": "structural_ablation_original_training_settings", "training": training, "metrics": metrics})
    return metrics








def read_structure_results(cfg, experiment="structure_v1"):
    root, dev = structure_output_directory(cfg, experiment), development(cfg)
    results = {}
    fingerprints = None
    for variant in ("no_mask", "no_attention"):
        folder = root / variant
        if not (folder / "metrics.json").is_file():
            continue
        if fingerprints is None:
            fingerprints = structure_input_fingerprints(cfg)
        training = json.loads((folder / "training_config.json").read_text(encoding="utf-8"))
        if training["input_files"] != fingerprints or training["variant"] != variant or training["experiment"] != experiment:
            raise ValueError(f"{variant}的结果与当前数据或实验名不同。")
        expected_contract = training.get("code_contract")
        if expected_contract is None:
            if training["notebook03_definitions_sha256"] != "3384c3746316d5dcf1e91757d4d75a5bb4f8386efaf603d4951ac9b4eaccccdd":
                raise ValueError("旧消融结果来自未知版本，请先核对实现。")
            expected_contract = LEGACY_STRUCTURE_CONTRACT
        if structure_code_contract(cfg) != expected_contract:
            raise ValueError(f"{variant}的网络或训练实现与结果版本不同。")
        table = pd.read_csv(folder / "development_oof.csv", dtype={"mr": str})
        validate_development_predictions(table, dev)
        metrics = json.loads((folder / "metrics.json").read_text(encoding="utf-8"))
        if not np.isclose(roc_auc_score(table.label, table.probability), metrics["pooled_auc"], atol=1e-12, rtol=0):
            raise ValueError(f"{variant}的汇总AUC与预测文件不一致。")
        results[variant] = {"metrics": metrics, "predictions": table}
    return results








def ablation_summary(cfg, experiment="structure_v1"):
    p = paths(cfg)
    rows = []
    baseline_path = p["temporal"] / "development_cv_metrics.json"
    if baseline_path.is_file():
        baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
        rows.append({"model": "三序列＋病灶＋注意力", "mean_fold_auc": baseline["mean_auc"],
            "std_fold_auc": baseline["std_auc"], "pooled_auc": baseline["pooled_auc"],
            **{k: baseline[k] for k in ("accuracy", "sensitivity", "specificity")}})
    single_summary = p["single"] / "summary.csv"
    if single_summary.is_file():
        names = {"t1": "T1-FLAIR＋病灶＋注意力", "t2": "T2＋病灶＋注意力", "dwi": "DWI＋病灶＋注意力"}
        for _, row in pd.read_csv(single_summary).iterrows():
            rows.append({"model": names[row.sequence], "mean_fold_auc": row.mean_fold_auc, "pooled_auc": row.auc,
                         **{k: row[k] for k in ("accuracy", "sensitivity", "specificity")}})
    for variant, result in read_structure_results(cfg, experiment).items():
        m = result["metrics"]
        rows.append({"model": "去病灶分支" if variant == "no_mask" else "去注意力（平均汇总）",
                     **{k: m[k] for k in ("mean_fold_auc", "std_fold_auc", "pooled_auc", "accuracy", "sensitivity", "specificity")}})
    return pd.DataFrame(rows)








def compare_ablation_oof(cfg, experiment="structure_v1"):
    p, dev = paths(cfg), development(cfg)
    baseline = pd.read_csv(p["temporal"] / "development_oof.csv", dtype={"mr": str}).rename(columns={"dev_oof_prob": "probability"})
    validate_development_predictions(baseline, dev)
    candidates = {}
    for sequence in ("t1", "t2", "dwi"):
        path = p["single"] / sequence / "development_oof.csv"
        if path.is_file():
            candidates[sequence] = pd.read_csv(path, dtype={"mr": str})
    candidates.update({name: result["predictions"] for name, result in read_structure_results(cfg, experiment).items()})
    rows = []
    for name, table in candidates.items():
        validate_development_predictions(table, dev)
        rows.append({"a": "完整三序列模型", "b": name, **paired_comparison(baseline, table)})
    return pd.DataFrame(rows)








## 1. 运行三组单序列训练

启用后会依次完成三个序列、每个序列五折的训练。结果写入 `single_sequence/t1/`、`t2/` 和 `dwi/`，训练结束后返回三组汇总指标。


In [ ]:
RUN_SINGLE_SEQUENCE = False

if RUN_SINGLE_SEQUENCE:
    summary = run_single_sequence_ablation(cfg)
    display(summary)


## 2. 运行两组结构消融

只做新增实验时，上一节 `RUN_SINGLE_SEQUENCE` 保持False。将下面需要运行的组改为True：每组会完成五折训练，两组都开启时依次运行，共十折。

`RUN_NO_MASK` 对应去病灶分支，`RUN_NO_ATTENTION` 对应去注意力并改为平均汇总。组名和输出目录固定区分，已有结果不会被覆盖。


In [ ]:
RUN_NO_MASK = False
RUN_NO_ATTENTION = False

if RUN_NO_MASK:
    no_mask_metrics = run_structure_ablation(cfg, "no_mask", experiment=STRUCTURE_EXPERIMENT)
    display(pd.Series(no_mask_metrics))

if RUN_NO_ATTENTION:
    no_attention_metrics = run_structure_ablation(cfg, "no_attention", experiment=STRUCTURE_EXPERIMENT)
    display(pd.Series(no_attention_metrics))


## 3. 汇总完整模型、单序列与结构消融

读取已有结果，不重新训练。表中同时展示五折平均AUC、合并OOF AUC以及阈值相关指标。原单序列汇总没有保存折间标准差，该列留空。

只有已完成的组会进入表中。新增结构组会核对患者、结局和结果来源，避免把不同数据或不同实验名下的结果混用。


In [ ]:
summary = ablation_summary(cfg, experiment=STRUCTURE_EXPERIMENT)
if len(summary):
    display(summary)
else:
    print("尚无可比较的开发结果。")


## 4. 比较完整模型与各消融组的开发AUC差值

把02完整模型设为A，分别与已完成的单序列和结构消融组B比较。函数按MR对齐同一批患者，差值为“A减B”；正值表示完整模型的合并OOF AUC更高。未运行的组会跳过。

配对bootstrap得到的是当前开发预测的描述性差异区间。各折曾用留出折AUC选择epoch，这些比较没有消除模型选择带来的偏差，也不替代时间验证。


In [ ]:
COMPARE_DEVELOPMENT_OOF = False
if COMPARE_DEVELOPMENT_OOF:
    differences = compare_ablation_oof(cfg, experiment=STRUCTURE_EXPERIMENT)
    display(differences)
    if len(differences):
        comparison_dir = structure_output_directory(cfg, STRUCTURE_EXPERIMENT)
        comparison_dir.mkdir(parents=True, exist_ok=True)
        differences.to_csv(comparison_dir / "development_auc_comparisons.csv", index=False)


## 结果保存与再次打开

新增两组分别保存到 `structure_ablation/<STRUCTURE_EXPERIMENT>/no_mask/` 和 `no_attention/`。每组包含五个折模型、`development_oof.csv`、`fold_metrics.csv`、`metrics.json` 和训练配置记录。原 `single_sequence/` 与 `temporal/` 的模型和预测文件继续保留。

再次打开时，将已完成组的运行开关保持False即可读取结果。如确实需要另跑一轮结构消融，可以更改上方 `STRUCTURE_EXPERIMENT`，不必改主配置的run_id；已有组不会被覆盖。中断后保存的部分折模型同样保留，当前版本不自动接续未完成的一组。

这里新增的是开发五折消融，没有另行生成全开发集重训模型或61例时间验证预测。比较时使用02的开发OOF基线，保持评估范围一致。通过去病灶组观察人工掩模分支的作用，通过去注意力组观察可学习切片权重相对平均汇总的作用。
